In [8]:
!pip install -q ultralytics pyyaml gdown

!gdown --fuzzy "https://drive.google.com/file/d/1CU2P9RuTDsHqutT8E96U8MS0lR26Jn59/view?usp=drive_link" -O /content/fridge_ds.zip
!ls -lh /content/fridge_ds.zip
!unzip -q /content/fridge_ds.zip -d /content/fridge_ds
!rm /content/fridge_ds.zip
!ls /content/fridge_ds

Downloading...
From (original): https://drive.google.com/uc?id=1CU2P9RuTDsHqutT8E96U8MS0lR26Jn59
From (redirected): https://drive.google.com/uc?id=1CU2P9RuTDsHqutT8E96U8MS0lR26Jn59&confirm=t&uuid=7a77fa02-7a84-4fbb-b65f-84657518a2bd
To: /content/fridge_ds.zip
100% 732M/732M [00:06<00:00, 113MB/s]
-rw-r--r-- 1 root root 698M Nov 22 13:59 /content/fridge_ds.zip
data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


In [9]:
import json, yaml
from pathlib import Path
from ultralytics import YOLO

ds_dir = Path("/content/fridge_ds")
orig_yaml = list(ds_dir.rglob("data.yaml"))[0]
root_ds = orig_yaml.parent

with open(orig_yaml, "r") as f:
    cfg = yaml.safe_load(f)
cfg["path"] = str(root_ds.resolve())
cfg["train"], cfg["val"], cfg["test"] = "train/images", "valid/images", "test/images"

fixed_yaml = root_ds / "data_resolved.yaml"
with open(fixed_yaml, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

model = YOLO("yolov8n.pt")
model.train(
    data=str(fixed_yaml),
    epochs=35,
    imgsz=640,
    batch=16,
    patience=8,
    device=0,
    project="/content/runs_fridge",
    name="yolo_fridge_v1",
)

best_pt = "/content/runs_fridge/yolo_fridge_v1/weights/best.pt"
metrics = YOLO(best_pt).val(data=str(fixed_yaml), split="test", device=0)

summary = {
    "model": "yolov8n",
    "epochs_configured": 35,
    "imgsz": 640,
    "mAP50": round(float(metrics.box.map50), 4),
    "mAP50_95": round(float(metrics.box.map), 4),
    "precision": round(float(metrics.box.mp), 4),
    "recall": round(float(metrics.box.mr), 4),
    "f1_approx": round(float(2 * metrics.box.mp * metrics.box.mr / (metrics.box.mp + metrics.box.mr + 1e-9)), 4),
    "speed_ms": {k: round(float(v), 2) for k, v in metrics.speed.items()},
}
with open("/content/yolo_metrics.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fridge_ds/data_resolved.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=35, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fracti

In [10]:
from google.colab import files
import shutil

# Empaqueta best.pt, yolo_metrics.json y la matriz de confusión / curvas PR para tu memoria
shutil.copy("/content/runs_fridge/yolo_fridge_v1/weights/best.pt", "/content/best.pt")
!zip -j /content/yolo_entrega.zip /content/best.pt /content/yolo_metrics.json \
  /content/runs_fridge/yolo_fridge_v1/results.png \
  /content/runs_fridge/yolo_fridge_v1/confusion_matrix_normalized.png
!cp /content/yolo_entrega.zip /content/drive/MyDrive/
files.download("/content/yolo_entrega.zip")

  adding: best.pt (deflated 9%)
  adding: yolo_metrics.json (deflated 40%)
  adding: results.png (deflated 7%)
  adding: confusion_matrix_normalized.png (deflated 21%)
cp: cannot create regular file '/content/drive/MyDrive/': No such file or directory


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>